# VisualGuard-LVLM on Kaggle
**Before running:** Session options -> Accelerator = GPU (T4 x2 or P100), Internet = On.
The COCO 2017 input is only needed for the optional CHAIR cell at the end.

## 1. Get the repo and install

In [1]:
import glob, os, shutil, subprocess, zipfile
from pathlib import Path

REPO = Path("/kaggle/working/VisualGuard-LVLM")

def first(*patterns):
    for pat in patterns:
        hits = sorted(glob.glob(pat))
        if hits:
            return hits[0]
    return None

if not REPO.exists():
    run_py = first(*[f"/kaggle/input/{'*/' * d}src/run.py" for d in range(6)])
    zp = first(*[f"/kaggle/input/{'*/' * d}VisualGuard*.zip" for d in range(6)])
    if run_py:
        shutil.copytree(Path(run_py).parent.parent, REPO)
    elif zp:
        with zipfile.ZipFile(zp) as z:
            z.extractall("/kaggle/working/_unz")
        shutil.copytree(next(Path("/kaggle/working/_unz").glob("*/src")).parent, REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/ria0304/VisualGuard-LVLM.git", str(REPO)], check=True)

os.chdir(REPO)
!pip install -q pycocotools sentencepiece pyarrow huggingface_hub accelerate "bitsandbytes>=0.43.0"

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

Cloning into '/kaggle/working/VisualGuard-LVLM'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.8 MB/s eta 0:00:00:00:0100:01
torch 2.11.0+cu128 | transformers 5.16.1 | cuda True Tesla T4


## 2. Fix for Grounding DINO with transformers 5.x (region rows D and F)

In [2]:
p = REPO / "src/model/grounding.py"
s = p.read_text()
old = '''        for score, label, box in zip(
            results["scores"].tolist(),
            results.get("text_labels", results.get("labels")).tolist(),
            results["boxes"].tolist(),
        ):'''
new = '''        raw_labels = results.get("text_labels", results.get("labels"))
        if hasattr(raw_labels, "tolist"):
            raw_labels = raw_labels.tolist()
        for score, label, box in zip(
            results["scores"].tolist(),
            list(raw_labels),
            results["boxes"].tolist(),
        ):'''
if old in s:
    p.write_text(s.replace(old, new)); print("patched")
else:
    print("already patched (or file differs)")

patched


## 3. Settings

In [3]:
MODEL = "llava-hf/llava-1.5-7b-hf"   # or "llava-hf/llava-interleave-qwen-0.5b-hf" for a fast test
QUANT = "4bit"                        # 7B fp16 (~14 GB) won't fit reliably on a T4/P100; use None for the 0.5B model
N     = 200                           # samples per POPE split; None = full splits (much slower)

QUANT_ARGS = f"--quantization {QUANT}" if QUANT else ""
MAXS       = f"--max-samples {N}" if N else ""
DATA = Path("/kaggle/working/POPE"); IMG = DATA / "images"
OUT  = "/kaggle/working/results_kaggle"   # fresh dir so the repo's old pilot results don't mix in

## 4. Unit tests (no downloads)

In [4]:
!python -m pytest tests -q

..........................................s............................. [ 22%]
........................................................................ [ 44%]
........................................................................ [ 66%]
........................................................................ [ 88%]
.....................................                                    [100%]
324 passed, 1 skipped in 3.04s


## 5. Download and convert POPE

In [5]:
from huggingface_hub import hf_hub_download
PQ = Path("/kaggle/working/pope_parquet"); PQ.mkdir(exist_ok=True)
for s in ("random", "popular", "adversarial"):
    p = hf_hub_download("lmms-lab/POPE", f"Full/{s}-00000-of-00001.parquet", repo_type="dataset")
    shutil.copy(p, PQ / f"{s}-00000-of-00001.parquet")

ROWS = f"--max-rows {N}" if N else ""
!python output/convert_pope.py --parquet-dir {PQ} --data-root {DATA} --image-root {IMG} {ROWS}

Full/random-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.0MB            

Full/random-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Full/popular-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.0MB            

Full/popular-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Full/adversarial-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.0MB            

Full/adversarial-00000-of-00001.parquet: downloading bytes:           |  0.00B            

random: wrote 200 rows -> /kaggle/working/POPE/coco_pope_random.jsonl
popular: wrote 200 rows -> /kaggle/working/POPE/coco_pope_popular.jsonl
adversarial: wrote 200 rows -> /kaggle/working/POPE/coco_pope_adversarial.jsonl
images -> /kaggle/working/POPE/images


## 6. Quick check (2 samples per row, numbers are meaningless)

In [6]:
!python scripts/run_ablations.py --smoke --benchmark pope --data-root {DATA} --image-root {IMG} \
    --model {MODEL} --device cuda --dtype float16 --output-dir /kaggle/working/smoke_results \
    --grounding-backend hf_grounding_dino -- {QUANT_ARGS}

Planned runs: 7
SMOKE MODE: 2 samples per split. These numbers are NOT comparable
to a full run and must not be reported as results.

RUNNING A. baseline  (method=baseline config=baseline)
  /usr/bin/python3 -m src.run --benchmark pope --method baseline --model llava-hf/llava-1.5-7b-hf --data-root /kaggle/working/POPE --output-dir /kaggle/working/smoke_results --seed 42 --run-name ablation_baseline --log-level WARNING --device cuda --dtype float16 --config baseline --image-root /kaggle/working/POPE/images --max-samples 2 --grounding-backend hf_grounding_dino --quantization 4bit
2026-10-15 18:20:09,236 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
config.json: 100%|█████████████████████████████| 950/950 [00:00<00:00, 4.78MB/s]
model.safetensors.index.json: 100%|█████████| 70.1k/70.1k [00:00<00:00, 178MB/s]
Reconstructing (incomplete total...): |            |  

## 7. Full ablation grid (A-G + lambda sweep)
If region rows still fail, add `--skip-rows D,F`.

In [7]:
!python scripts/run_ablations.py --benchmark pope --data-root {DATA} --image-root {IMG} \
    --model {MODEL} --device cuda --dtype float16 {MAXS} --output-dir {OUT} \
    --lambda-sweep 0.25,0.5,1.0 --grounding-backend hf_grounding_dino -- {QUANT_ARGS}

Planned runs: 10

RUNNING A. baseline  (method=baseline config=baseline)
  /usr/bin/python3 -m src.run --benchmark pope --method baseline --model llava-hf/llava-1.5-7b-hf --data-root /kaggle/working/POPE --output-dir /kaggle/working/results_kaggle --seed 42 --run-name ablation_baseline --log-level WARNING --device cuda --dtype float16 --config baseline --image-root /kaggle/working/POPE/images --max-samples 200 --grounding-backend hf_grounding_dino --quantization 4bit
2026-10-15 18:25:35,601 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 3 files: 100%|█████████████████████████| 3/3 [00:00<00:00, 1162.61it/s]
Download complete: :                                        |  0.00B            
Download complete: :                                        |  0.00B            
Reco

## 8. Sampling and beam baselines + results table

In [3]:
import glob, os, shutil, subprocess, zipfile
from pathlib import Path

REPO = Path("/kaggle/working/VisualGuard-LVLM")

def first(*patterns):
    for pat in patterns:
        hits = sorted(glob.glob(pat))
        if hits:
            return hits[0]
    return None

if not REPO.exists():
    run_py = first(*[f"/kaggle/input/{'*/' * d}src/run.py" for d in range(6)])
    zp = first(*[f"/kaggle/input/{'*/' * d}VisualGuard*.zip" for d in range(6)])
    if run_py:
        shutil.copytree(Path(run_py).parent.parent, REPO)
    elif zp:
        with zipfile.ZipFile(zp) as z:
            z.extractall("/kaggle/working/_unz")
        shutil.copytree(next(Path("/kaggle/working/_unz").glob("*/src")).parent, REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/ria0304/VisualGuard-LVLM.git", str(REPO)], check=True)
os.chdir(REPO)
!pip install -q pycocotools sentencepiece pyarrow huggingface_hub accelerate "bitsandbytes>=0.43.0"

# --- patch 1: Grounding DINO labels (transformers 5.x) ---
p = REPO / "src/model/grounding.py"
s = p.read_text()
old = '''        for score, label, box in zip(
            results["scores"].tolist(),
            results.get("text_labels", results.get("labels")).tolist(),
            results["boxes"].tolist(),
        ):'''
new = '''        raw_labels = results.get("text_labels", results.get("labels"))
        if hasattr(raw_labels, "tolist"):
            raw_labels = raw_labels.tolist()
        for score, label, box in zip(
            results["scores"].tolist(),
            list(raw_labels),
            results["boxes"].tolist(),
        ):'''
print("grounding:", "patched" if old in s else "not found")
p.write_text(s.replace(old, new))

# --- patch 2: beam length_penalty=None ---
p = REPO / "src/model/visual_guard_decoder.py"
s = p.read_text()
old = '''                num_beams=self.decoding_config.num_beams,
                early_stopping=True,
            )'''
new = '''                num_beams=self.decoding_config.num_beams,
                early_stopping=True,
                length_penalty=(
                    1.0
                    if self.decoding_config.length_penalty is None
                    else self.decoding_config.length_penalty
                ),
            )'''
print("beam:", "patched" if old in s else "not found")
p.write_text(s.replace(old, new))

# --- settings ---
MODEL = "llava-hf/llava-1.5-7b-hf"
QUANT = "4bit"
N     = 200
QUANT_ARGS = f"--quantization {QUANT}" if QUANT else ""
MAXS       = f"--max-samples {N}" if N else ""
ROWS       = f"--max-rows {N}" if N else ""
DATA = Path("/kaggle/working/POPE"); IMG = DATA / "images"
OUT  = "/kaggle/working/results_kaggle"

Cloning into '/kaggle/working/VisualGuard-LVLM'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 50.0 MB/s eta 0:00:00:00:0100:01
grounding: patched
beam: patched


In [4]:
p = REPO / "src/model/visual_guard_decoder.py"
s = p.read_text()
old = '''                num_beams=self.decoding_config.num_beams,
                early_stopping=True,
            )'''
new = '''                num_beams=self.decoding_config.num_beams,
                early_stopping=True,
                length_penalty=(
                    1.0
                    if self.decoding_config.length_penalty is None
                    else self.decoding_config.length_penalty
                ),
            )'''
print("patched" if old in s else "already patched / differs")
p.write_text(s.replace(old, new))

already patched / differs


37189

In [5]:
for m in ["sampling", "beam"]:
    !python -m src.run --benchmark pope --method {m} --model {MODEL} --device cuda --dtype float16 \
        --data-root {DATA} --image-root {IMG} {MAXS} --output-dir {OUT} {QUANT_ARGS}

!python scripts/make_results_table.py --results-dir {OUT} | tee {OUT}/results_table.md

2026-10-16 04:59:20,310 INFO    visualguard: Method: sampling
2026-10-16 04:59:20,311 INFO    visualguard: LVLM: llava-hf/llava-1.5-7b-hf
2026-10-16 04:59:20,311 INFO    visualguard: Evidence weights: alpha=0.0 beta=0.0 gamma=0.0 lambda=0.0 threshold=1.0
2026-10-16 04:59:20,311 ERROR   visualguard: Dataset problem: POPE annotation file not found: /kaggle/working/POPE/coco_pope_random.jsonl
Expected coco_pope_random.jsonl inside /kaggle/working/POPE (official POPE release).
2026-10-16 04:59:26,393 INFO    visualguard: Method: beam
2026-10-16 04:59:26,394 INFO    visualguard: LVLM: llava-hf/llava-1.5-7b-hf
2026-10-16 04:59:26,394 INFO    visualguard: Evidence weights: alpha=0.0 beta=0.0 gamma=0.0 lambda=0.0 threshold=1.0
2026-10-16 04:59:26,394 ERROR   visualguard: Dataset problem: POPE annotation file not found: /kaggle/working/POPE/coco_pope_random.jsonl
Expected coco_pope_random.jsonl inside /kaggle/working/POPE (official POPE release).
No result JSON files in /kaggle/working/results_

## 9. (Optional) CHAIR captioning with the COCO 2017 input
Uses val2017, so numbers are not comparable to published CHAIR (val2014).

In [8]:
import os
print("top-level inputs:", os.listdir("/kaggle/input"))

for root, dirs, files in os.walk("/kaggle/input"):
    depth = root[len("/kaggle/input"):].count(os.sep)
    if depth >= 4:
        dirs[:] = []          # don't descend further
        continue
    if "instances_val2017.json" in files or os.path.basename(root) == "val2017":
        print("FOUND:", root)
    dirs[:] = [d for d in dirs if d not in ("train2017", "test2017")]  # skip the huge folders

top-level inputs: ['datasets', 'models']


In [9]:
ANN  = first(*[f"/kaggle/input/{'*/' * d}annotations/instances_val2017.json" for d in range(9)])
CIMG = first(*[f"/kaggle/input/{'*/' * d}val2017" for d in range(9)])
print("ANN :", ANN)
print("CIMG:", CIMG)
assert ANN and CIMG, "COCO not found"
ANN_DIR = str(Path(ANN).parent)
SYN = "/kaggle/working/synonyms.txt"
!wget -q -O {SYN} https://raw.githubusercontent.com/LisaAnne/Hallucination/master/data/synonyms.txt

for m, cfg in [("baseline", ""), ("vcd", ""), ("visualguard", "--config visualguard")]:
    !python scripts/run_chair.py --method {m} {cfg} --model {MODEL} --device cuda --dtype float16 {QUANT_ARGS} \
        --ann-dir {ANN_DIR} --image-root {CIMG} --coco-split val2017 --synonyms {SYN} \
        --num-images 100 --max-new-tokens 128 --output-dir {OUT}

ANN : /kaggle/input/datasets/sabahesaraki/2017-2017/annotations_trainval2017/annotations/instances_val2017.json
CIMG: /kaggle/input/datasets/sabahesaraki/2017-2017/val2017
2026-10-16 05:04:10,369 INFO    numexpr.utils: NumExpr defaulting to 4 threads.
2026-10-16 05:04:10,754 INFO    src.model.llava_backend: Loading LVLM llava-hf/llava-1.5-7b-hf (dtype=torch.float16, device=cuda, quant=4bit)
2026-10-16 05:04:10,932 INFO    httpx: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-10-16 05:04:10,992 INFO    httpx: HTTP Request: HEAD https://huggingface.co/llava-hf/llava-1.5-7b-hf/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-16 05:04:10,993 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-16 05:04:11,009 INFO    httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/llava-hf/llava-1.5

## 10. Bundle results for download

In [13]:
shutil.make_archive("/kaggle/working/visualguard_results", "zip", OUT)

'/kaggle/working/visualguard_results.zip'

In [14]:
from IPython.display import FileLink
FileLink("/kaggle/working/visualguard_results.zip")

/kaggle/working/visualguard_results.zip